In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

In [5]:
# Encode image and send to vision model
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(
    # model="openai/gpt-oss-20b",
    model="qwen/qwen3.8-27b",
    temperature=0,
)

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

Based on the blood work report provided for **Rajesh Sharma (Age 48, Male)**, here is the extraction of all test results with flags for values outside the normal range.

### **COMPLETE BLOOD COUNT (CBC)**
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| **Hemoglobin** | 15.1 g/dL | 13.5 – 17.5 | Normal |
| **Hematocrit** | 44% | 41 – 53% | Normal |
| **WBC** | 6.8 x 10³/uL | 4.5 – 11.0 | Normal |
| **Platelets** | 220 x 10³/uL | 150 – 400 | Normal |

### **LIPID PANEL**
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| **Total Cholesterol** | 238 mg/dL | < 200 | **HIGH** |
| **LDL Cholesterol** | 162 mg/dL | < 100 | **HIGH** |
| **HDL Cholesterol** | 36 mg/dL | > 40 | **LOW** |
| **Triglycerides** | 188 mg/dL | < 150 | **HIGH** |

### **METABOLIC PANEL**
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| **Glucose (Fasting)** | 92 mg/dL | 70 – 99 | Normal |
| **HbA1c** | 5.3% | < 5.7% | Normal |
| **Creatinine** | 

### Suggest Diet Plan Agent

In [6]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [ ]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
Output in a 2-3 lines max
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [10]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

### Diet Plan for High Cholesterol

**Recommended Foods (Eat):**
*   Fruits
*   Vegetables
*   Whole grains
*   Lean protein

**Foods to Avoid (Do Not Eat):**
*   Red meat
*   Fried food
*   Full-fat dairy
*   Processed snacks

This diet focuses on reducing saturated fats and increasing fiber intake to help lower LDL ("bad") cholesterol and triglycerides while supporting overall heart health.
